In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import numpy as np

# Ler a tabela Gold
df_ts = spark.table("workspace.default.gold_series_temporais_pedidos_hora")

# Correção: Particionar por hora e ordenar por dia (lag(1) pega a mesma hora no dia anterior)
window_spec = Window.partitionBy("hora").orderBy("data_dia")

df_baseline = df_ts \
    .withColumn("receita_baseline", F.lag("receita", 1).over(window_spec)) \
    .withColumn("pedidos_baseline", F.lag("num_pedidos", 1).over(window_spec))

# Filtrar para o período da tarde (12h às 18h) e remover nulos do primeiro dia
df_tarde = df_baseline.filter((F.col("hora") >= 12) & (F.col("hora") <= 18)) \
    .filter(F.col("receita_baseline").isNotNull())

# Converter para Pandas para calcular as métricas exatas
pdf = df_tarde.select("receita", "receita_baseline", "num_pedidos", "pedidos_baseline").toPandas()

# Métricas Receita
y_true_rec, y_pred_rec = pdf["receita"].values, pdf["receita_baseline"].values
mae_rec = np.mean(np.abs(y_true_rec - y_pred_rec))
rmse_rec = np.sqrt(np.mean((y_true_rec - y_pred_rec) ** 2))
wape_rec = np.sum(np.abs(y_true_rec - y_pred_rec)) / np.sum(y_true_rec)

# Métricas Pedidos
y_true_ped, y_pred_ped = pdf["num_pedidos"].values, pdf["pedidos_baseline"].values
mae_ped = np.mean(np.abs(y_true_ped - y_pred_ped))
rmse_ped = np.sqrt(np.mean((y_true_ped - y_pred_ped) ** 2))
wape_ped = np.sum(np.abs(y_true_ped - y_pred_ped)) / np.sum(y_true_ped)

print(f"=== MÉTRICAS DO BASELINE CORRIGIDO (12h - 18h) ===")
print(f"Receita -> MAE: R$ {mae_rec:.2f} | RMSE: R$ {rmse_rec:.2f} | WAPE: {wape_rec*100:.2f}%")
print(f"Pedidos -> MAE: {mae_ped:.2f} | RMSE: {rmse_ped:.2f} | WAPE: {wape_ped*100:.2f}%")